This notebook is used to evaluate the models trained on the KSE data. 
Set a folder of weights files in WEIGHTS_FOLDER or specify a specific list of files by setting WEIGHTS_FILES. 
Make sure to include 'model_weights/' if you are setting WEIGHTS_FILES manually. 

If you would like to save plots and mode evolution data, change SAVE_PICS to true. 
Otherwise, the plots will simply be displayed sequentially in the notebook.
Note that previous files of the same name will not be overwritten unless OVERWRITE is set to True.

The SIMPLY_LOAD variable is there in case you want to generate plots based on previously generated & saved predictions (simply load data instead of making predictions from scratch).

In [ ]:
# dependencies
from pathlib import Path
import re
import logging
from IPython.display import display, Image 
from functools import cached_property
import ipywidgets as widgets

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy import stats
from scipy.stats import wasserstein_distance

import torch
import torch.nn as nn

# Set up standard logging instead of raw prints/warnings
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# global variables
WEIGHTS_FOLDER = 'trainer9-realsp-trial1'  # ex: 'trainer7-fouriersp-normalinit-0.01std-t1/'
if WEIGHTS_FOLDER is None:
    WEIGHTS_FILES = [
        'model_weights/trainer9-realsp-trial1/epoch001.pth',
        'model_weights/trainer9-realsp-trial1/epoch002.pth',
    ]
else:
    weights_dir = Path('model_weights') / WEIGHTS_FOLDER
    WEIGHTS_FILES = [
        str(weights_dir / f.name)
        for f in weights_dir.iterdir()
        if f.is_file() and 'latest-checkpoint' not in f.name
    ]

SAVE_PICS = True
OVERWRITE = True
SIMPLY_LOAD = False
CALC_BLOWUP = False
EPOCHS_TO_SHOW = None  # this gets set later on

# variable maps
series_length_map = {
    'complete': 40000,
    'nouux': 160,
    'nouxx': 48,
    'nouxxxx': 6
}
activation_map = {
    'sigmoid': nn.Sigmoid,
    'gelu': nn.GELU,
    'relu': nn.ReLU,
    'tanh': nn.Tanh
}
initialization_map = {
    'gaussian': 0,
    'normal': 0,
    'kaiming_normal': 1,
    'orthogonal': 2
}
inv_initialization_map = {v: k for k, v in initialization_map.items()}

In [ ]:
# helper functions

def fullFourier(csv_file):
    """
    converts a csv file will alternating columns of real and imaginary numbers into a complex-valued numpy array
    assumes that the csv file represents the 1st--nth modes of a fourier representation, so adds zeros for 1st & n+1th mode and the flipped complex conjugates for the n+2th -- 2nth modes
    """
    # read data
    raw_data = pd.read_csv(csv_file, header=None)
    raw_data = raw_data.to_numpy()

    # turn into complex representation
    complexified = raw_data[:, 0::2] + raw_data[:, 1::2]*1j

    # create list of zeros to be added
    if raw_data.shape[1] % 2 == 0: 
        ceros = np.zeros((len(raw_data), 1), dtype=complex)
    else: raise TypeError("Input must have an even number of columns")

    # use hermitian symmetry to create the full fourier representation
    output = np.hstack((
        ceros,
        complexified,
        ceros,
        np.flipud(np.conj(complexified)),
    ))

    return output

def ksfm2real(a, L, n=None):
    """
    Convert Fourier mode representation a(k, t) to real space u(x, t)
    Adapted from code in http://ChaosBook.org/
    Inputs
    a : ndarray, shape (m, nt)
        Fourier mode representation where rows alternate:
        [Re(v1), Im(v1), Re(v2), Im(v2), ...]
    L : float
        Domain length.
    n : int, optional
        Number of spatial grid points (excluding duplicated endpoint).

    Outputs
    x : ndarray, shape (n+1,)
        Spatial coordinates.
    u : ndarray, shape (n+1, nt)
        Solution in real space.
    ux : ndarray, shape (n+1, nt)
        First derivative.
    uxx : ndarray, shape (n+1, nt)
        Second derivative.
    """

    if n is None:
        n = a.shape[0] + 2
    if n < a.shape[0] + 2:
        n = a.shape[0] + 2

    nt = a.shape[1]

    # Spatial grid
    x = L * np.arange(-n/2, n/2 + 1) / n

    # convert alternating real-imaginary representation to complex representation of fourier coefficients
    v = a[0::2, :] + 1j * a[1::2, :]

    # corresponds to the zeroeth mode and the middle mode, bc middle is alone in discrete fft
    zeros1 = np.zeros((1, nt), dtype=complex)
    zeros2 = np.zeros((n - a.shape[0] - 1, nt), dtype=complex)

    vv = np.vstack([
        zeros1,
        v,
        zeros2,
        np.flipud(np.conj(v)) # flip bc Hermetian symmetry
    ])

    u = np.real(np.fft.fft(vv, axis=0))
    #u = np.vstack([u, u[0:1]]) idk why they have this it just adds the first row again

    # First derivative
    ik = -(2j * np.pi / L) * np.arange(1, v.shape[0] + 1)[:, None]
    vx = ik * v

    vv = np.vstack([
        zeros1,
        vx,
        zeros2,
        np.flipud(np.conj(vx))
    ])

    ux = np.real(np.fft.fft(vv, axis=0))

    # Second derivative
    vxx = ik * vx

    vv = np.vstack([
        zeros1,
        vxx,
        zeros2,
        np.flipud(np.conj(vxx))
    ])

    uxx = np.real(np.fft.fft(vv, axis=0))

    return x, u, ux, uxx

def slideshow(epoch_to_image_path, epochs_to_show):
    """
    Creates a slideshow out of saved images corresponding to the relevant models
    """

    # used to actually display the loaded image
    def display_saved_plot(chosen_epoch):
        img_path = epoch_to_image_path[chosen_epoch]
        display(Image(filename=img_path))

    # --- UI Layout Elements ---
    epoch_slider = widgets.SelectionSlider(
        options=epochs_to_show,        
        value=epochs_to_show[0],       
        description='Epoch:',
        continuous_update=True
    )

    # button commands
    def on_prev_clicked(b):
        current_index = epochs_to_show.index(epoch_slider.value)
        if current_index > 0:
            epoch_slider.value = epochs_to_show[current_index - 1]

    def on_next_clicked(b):
        current_index = epochs_to_show.index(epoch_slider.value)
        if current_index < len(epochs_to_show) - 1:
            epoch_slider.value = epochs_to_show[current_index + 1]

    # making and coding the buttons
    prev_button = widgets.Button(description='', icon='arrow-left', layout=widgets.Layout(width='50px'))
    next_button = widgets.Button(description='', icon='arrow-right', layout=widgets.Layout(width='50px'))

    prev_button.on_click(on_prev_clicked)
    next_button.on_click(on_next_clicked)

    # link the widget to display_saved_plot() function which actually displays the image
    plot_output = widgets.interactive_output(display_saved_plot, {'chosen_epoch': epoch_slider})

    # Assemble the UI
    ui = widgets.VBox([
        widgets.HBox([prev_button, epoch_slider, next_button]), 
        plot_output
    ])

    display(ui)

def calc_wasserstein_mode_thresholds(ignore_first=0, window_rad=50, step=20, percentile=99.9, multiplier=1.2):
    """
    Computes a threshold based on the distribution of Wasserstein distances
    within the steady-state ground truth time series.
    
    Parameters:
    -----------
    ground_truth : np.ndarray
        Ground truth time series array.
    ignore_first : int
        Initial transient steps to ignore.
    window_rad : int
        Half window size for local distribution windows.
    step : int
        Stride for sliding the calibration window.
    percentile : float
        Percentile of GT distances to set as the baseline limit (e.g., 99.0 or 99.9).
    multiplier : float
        Safety factor applied to the computed percentile threshold.
        
    Returns:
    --------
    float : Computed distance threshold for runaway detection.
    """

    ground_truth_mode_evolution = pd.read_hdf('../mode_evolution_csvs/mode_amplitude_histories/ground_state.h5', key='df').to_numpy()
    
    num_modes = ground_truth_mode_evolution.shape[1]
    mode_thresholds = np.zeros(num_modes)

    for j in range(num_modes):
        gt_modej = ground_truth_mode_evolution[:, j]
        gt_steady = gt_modej[ignore_first:]

        gt_distances = []
        
        # Slide across ground truth to measure natural internal fluctuations
        for tstep in range(max(ignore_first, window_rad), len(gt_modej) - window_rad, step):
            gt_window = gt_modej[tstep - window_rad : tstep + window_rad]
            w_dist = wasserstein_distance(gt_window, gt_steady)
            gt_distances.append(w_dist)
            
        if not gt_distances:
            raise ValueError("Ground truth series is too short for the given window_rad and ignore_first.")
            
        # Calculate threshold based on chosen upper percentile + safety buffer
        base_threshold = np.percentile(gt_distances, percentile)
        calibrated_threshold = base_threshold * multiplier

        mode_thresholds[j] = calibrated_threshold

    return mode_thresholds

def find_attractor_runaway_wasserstein(predictions, ground_truth, ignore_first=0, window_rad=50, step=20, distance_threshold=None):
    """
    Detects mode runaway using 1D Wasserstein Distance to compare local 
    prediction windows against steady-state Ground Truth distributions.
    """
    # extract ground truth steady-state distribution
    gt_steady = ground_truth[ignore_first:]
    
    # establish baseline threshold if not explicitly passed
    if distance_threshold is None:
        # Split ground truth into two halves to measure natural internal distance
        mid = len(gt_steady) // 2
        gt_a, gt_b = gt_steady[:mid], gt_steady[mid:]
        baseline_dist = wasserstein_distance(gt_a, gt_b)
        
        # Scale baseline distance to establish a runaway cutoff
        distance_threshold = baseline_dist * 3.5 

    # slide window over PREDICTIONS
    start_idx = max(ignore_first, window_rad)
    end_idx = len(predictions) - window_rad

    for tstep in range(start_idx, end_idx, step):
        window = predictions[tstep - window_rad : tstep + window_rad]
        
        # Compute Wasserstein distance between local window and GT steady state
        w_dist = wasserstein_distance(window, gt_steady)

        # Trigger detection if distribution shifts beyond cutoff
        if w_dist > distance_threshold:
            return tstep

    return len(predictions)

def find_attractor_runaway_ztest(
    predictions,
    ground_truth,
    ignore_first=0,
    window_rad=50,
    step=20,
    z_threshold=3.0,
):
    """
    Detects when a predicted mode leaves the true attractor state space using one-way z-test, ignoring initial transient behavior prior to `ignore_first` timesteps.

    Parameters:
    -----------
    predictions : np.ndarray (shape: [T,])
        Predicted time series for a single mode.
    ground_truth : np.ndarray (shape: [T,])
        Ground truth time series for a single mode.
    ignore_first : int
        Number of initial time steps to ignore (allowing GT to reach steady state).
    window_rad : int
        Half window size for smoothing local statistics.
    step : int
        Stride size for window sliding.
    z_threshold : float
        Number of standard deviations above steady-state GT mean to count as runaway.
    """
    # Slice ground truth after ignore_first to represent steady-state attractor only
    gt_steady = ground_truth[ignore_first:]

    # Compute steady-state attractor statistics
    gt_mean = np.mean(gt_steady)
    gt_std = np.std(gt_steady)
    gt_std = max(gt_std, 1e-10) # Floor for std dev to prevent division by zero in zero-amplitude modes

    # Slide window over PREDICTIONS starting after ignore_first
    for tstep in range(max(ignore_first, window_rad), len(predictions) - window_rad, step):
        window = predictions[tstep - window_rad : tstep + window_rad]

        # Check statistical drift relative to steady-state attractor
        window_mean = np.mean(window)
        z_score = (window_mean - gt_mean) / gt_std

        if z_score > z_threshold:
            return tstep

    return len(predictions)  # Remained on the steady-state attractor

In [ ]:
# DEFINE MODEL ARCHITECTURE


class neuralODE9(nn.Module):
    def __init__(self, act_func='sigmoid', dim_inout=64, init_type='normal', init_params=[0.0, 0.01]):
        super().__init__()

        # define the string and nn module versions of the activation function
        self.act_func = act_func
        self.act_layer = activation_map[self.act_func]

        # do stuff with the model initialization parameters so that they can be saved to the register buffer later
        self.init_record = torch.tensor([initialization_map[init_type]])
        if self.init_record.item() == 0:
            self.init_record = torch.cat((self.init_record, torch.tensor(init_params)))

        # save model parameters to the register buffer for access during model evaluation/ plotting
        self.register_buffer('layer_init_record', self.init_record)
        self.register_buffer('data_mean', torch.zeros(dim_inout))  # placeholder mean
        self.register_buffer('data_std', torch.ones(dim_inout))  # placeholder stdev

        # use a linear & sigmoid fully connected nn
        self.net = nn.Sequential(
            nn.Linear(dim_inout, 200),
            self.act_layer(),
            nn.Linear(200, 200),
            self.act_layer(),
            nn.Linear(200, 200),
            self.act_layer(),
            nn.Linear(200, dim_inout)
        )
        self.init_weights()

    # if training data is normalized, save the real mean and standard deviation of the dataset
    def set_normalization_stats(self, mean, std):
        self.data_mean = mean
        self.data_std = std

    # initialize model weights & biases
    def init_weights(self):
        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                if self.init_record[0] == 0:
                    # initialization for random normal weights
                    nn.init.normal_(m.weight, mean=self.init_record[1], std=self.init_record[2])
                elif self.init_record[0] == 1:
                    # initialization for kaiming normal weights
                    nn.init.kaiming_normal_(m.weight, nonlinearity=self.act_func)
                elif self.init_record[0] == 2:
                    # initialization for orthogonal weights
                    nn.init.orthogonal_(m.weight)
                nn.init.constant_(m.bias, 0.0)

    # forward pass adapted to input/output numpy arrays instead of torch tensors
    def forward(self, t, x):
        x = torch.as_tensor(x, dtype=torch.float32, device=next(self.parameters()).device)
        return self.net(x).detach().cpu().numpy()

In [ ]:
# DEFINE MODEL WRAPPER OBJECT FOR OOP-based access of model & metadata later


class ModelWrapper:
    def __init__(self, weights_file: str, simply_load: bool, overwrite: bool = False):
        self.weights_file = Path(weights_file)
        # the clean weights file doesn't include the file extension or the model_weights superfolder
        self.clean_weights_file = Path(Path(weights_file).parent.name) / Path(weights_file).stem

        self.simply_load = simply_load
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # declare wrapper properties based on file name
        metadata = self._parse_metadata()
        self.trainer_version = metadata["trainer_version"]
        self.activation_func = metadata["activation_func"]
        self.equation = metadata["equation"]
        self.batch_size = metadata["batch_size"]
        self.initialization_type = metadata["initialization_type"]
        self.std = metadata["std"]
        self.training_space = metadata["training_space"]
        self.normalized = metadata["normalized"]
        self.trial = metadata["trial"]
        self.epoch = metadata["epoch"]

        self.training_data_file = Path("training_data") / ("u_hist" if self.training_space == "real" else "uhat_hist")
        if self.equation != "complete":
            self.training_data_file = self.training_data_file.with_name(
                f"{self.training_data_file.name}_{self.equation}.h5"
            )
        else:
            self.training_data_file = self.training_data_file.with_suffix(".h5")

        self.num_features = pd.read_hdf(self.training_data_file, key="df").to_numpy().shape[1]

        self.model = self._initialize_and_load_model()
        self.save_dirs = self._make_save_dirs(overwrite=overwrite if SAVE_PICS else True)

        self._predhat_split = None
        self._mode_amplitude_hist = None
        self._blow_up_tsteps = None
        self._mode_moving_average = None
        self._mode_moving_std = None
        self._us = None
        self._u_xs = None
        self._u_xxs = None

    def _parse_metadata(self):
        """Helper method to parse the metadata from the file name."""
        file_path = self.clean_weights_file.as_posix()

        # decleare defaults for all metadata
        meta = {
            "trainer_version": 9,
            "equation": "complete",
            "activation_func": "sigmoid",
            "batch_size": 75,
            "initialization_type": "normal",
            "std": 0.01,
            "training_space": "",
            "normalized": False,
            "trial": 0,
            "epoch": 0,
        }

        if "trainer9" in file_path:
            meta["trainer_version"] = 9

        if "eq" in file_path:
            if "nouuxeq" in file_path:
                meta["equation"] = "nouux"
            elif "nouxxeq" in file_path:
                meta["equation"] = "nouxx"
            elif "nouxxxxeq" in file_path:
                meta["equation"] = "nouxxxx"

        if "act" in file_path:
            if "geluact" in file_path:
                meta["activation_func"] = "gelu"
            elif "reluact" in file_path:
                meta["activation_func"] = "relu"
            elif "tanhact" in file_path:
                meta["activation_func"] = "tanh"

        if "bt" in file_path:
            split_file_path = re.split(r"[-/]+", file_path)
            relevant_part = next(filter(lambda x: "bt" in x, split_file_path), None)
            if relevant_part is not None:
                meta["batch_size"] = int(re.findall(r"\d+", relevant_part)[0])

        if "init" in file_path:
            if "kaiming_normalinit" in file_path:
                meta["initialization_type"] = "kaiming_normal"
            elif "orthogonalinit" in file_path:
                meta["initialization_type"] = "orthogonal"

        if "std" in file_path:
            split_file_path = re.split(r"[-/]+", file_path)
            relevant_part = next(filter(lambda x: "std" in x, split_file_path), None)
            if relevant_part is not None:
                meta["std"] = float(re.findall(r"\d+", relevant_part)[0])

        if "sp" in file_path:
            if "realsp" in file_path:
                meta["training_space"] = "real"
            elif "fouriersp" in file_path:
                meta["training_space"] = "fourier"
        else:
            logger.warning(f"Required training space missing from path: {file_path}")

        if "normed" in file_path:
            meta["normalized"] = True

        if "trial" in file_path:
            split_file_path = re.split(r"[-/]+", file_path)
            relevant_part = next(filter(lambda x: "trial" in x, split_file_path), None)
            if relevant_part is not None:
                meta["trial"] = int(re.findall(r"\d+", relevant_part)[0])
        else:
            logger.warning(f"Couldn't parse trial number from file path {file_path}")

        if "epoch" in file_path:
            split_file_path = re.split(r"[-/]+", file_path)
            meta["epoch"] = int(re.findall(r"\d+", split_file_path[-1])[0])
        else:
            logger.warning(f"Couldn't parse epoch number from file path: {file_path}")

        return meta

    def _initialize_and_load_model(self):
        """Instantiates the ODEModel and loads the weights into it."""
        # make sure the file actually exists
        if not self.weights_file.exists():
            raise FileNotFoundError(f"No weights file found at {self.weights_file}")

        if self.trainer_version == 9:
            model = neuralODE9(
                act_func=self.activation_func,
                dim_inout=self.num_features,
                init_type=self.initialization_type,
                init_params=[0.0, 0.01],
            )
        else:
            logger.warning(
                f"Couldn't make model {self.clean_weights_file} because it is an unsupported trainer version."
            )
            raise ValueError(f"Unsupported trainer version: {self.trainer_version}")

        try:
            state_dict = torch.load(self.weights_file, map_location=self.device)
            model.load_state_dict(state_dict)
        except Exception as e:
            logger.error(f"Failed to load state dict for {self.weights_file}")
            raise e

        model.to(self.device)
        model.eval()
        return model

    @property
    def predhat_split(self):
        """Lazy loading prediction."""
        if self._predhat_split is None:
            if self.simply_load:
                self._predhat_split = pd.read_hdf(self.save_dirs["predhat_split_hists"], key="df").to_numpy()
            else:
                self._predhat_split = self._make_predictions()
        return self._predhat_split

    def _make_predictions(self):
        """Makes predictions about the K-S system based on the model."""
        initial_condition = pd.read_hdf(self.training_data_file, key="df").iloc[0].to_numpy()

        t_f = series_length_map[self.equation]
        ts_eval = np.arange(0.0, t_f, 0.25)

        if self.normalized:
            mean = self.model.data_mean.detach().cpu().numpy()
            std = self.model.data_std.detach().cpu().numpy()
            initial_condition_normalized = (initial_condition - mean) / std

            if self.training_space == "fourier":
                predhat_normalized = solve_ivp(
                    self.model.forward,
                    [0, ts_eval[-1]],
                    initial_condition_normalized,
                    t_eval=ts_eval,
                )
                predhat_normalized = predhat_normalized.y.T
                return (predhat_normalized * std) + mean

            if self.training_space == "real":
                pred_normalized = solve_ivp(
                    self.model.forward,
                    [0, ts_eval[-1]],
                    initial_condition_normalized,
                    t_eval=ts_eval,
                )
                pred_normalized = pred_normalized.y.T

                pred = (pred_normalized * std) + mean
                predhat = np.fft.rfft(pred, axis=1, norm="forward")
                predhat_split = np.empty((predhat.shape[0], predhat.shape[1] * 2 - 4), dtype=np.float64)
                predhat_split[:, 0::2] = predhat.real[:, 1:-1]
                predhat_split[:, 1::2] = predhat.imag[:, 1:-1]
                return predhat_split

        if self.training_space == "fourier":
            predhat_split = solve_ivp(self.model.forward, [0, ts_eval[-1]], initial_condition, t_eval=ts_eval)
            return predhat_split.y.T

        if self.training_space == "real":
            pred = solve_ivp(self.model.forward, [0, ts_eval[-1]], initial_condition, t_eval=ts_eval)
            pred = pred.y.T

            predhat = np.fft.rfft(pred, axis=1, norm="forward")
            predhat_split = np.empty((predhat.shape[0], predhat.shape[1] * 2 - 4), dtype=np.float64)
            predhat_split[:, 0::2] = predhat.real[:, 1:-1]
            predhat_split[:, 1::2] = predhat.imag[:, 1:-1]
            return predhat_split

        raise ValueError(f"Unsupported training space: {self.training_space}")

    @property
    def mode_amplitude_hist(self):
        """Lazy loading mode amplitude history."""
        if self._mode_amplitude_hist is None:
            if self.simply_load:
                self._mode_amplitude_hist = pd.read_hdf(self.save_dirs["mode_amplitude_hists"], key="df").to_numpy()
            else:
                self._mode_amplitude_hist = self._calc_modes()
        return self._mode_amplitude_hist

    def _calc_modes(self):
        """Calculates the evolution of Fourier modes over time."""
        tsteps_range = np.array([0, series_length_map[self.equation]], dtype=int)
        columns_to_plot = np.arange(0, 62, 2, dtype=int)

        mode_amplitude_hist = np.zeros((tsteps_range[1]-tsteps_range[0], 31))
        
        # calculate the mode amplitudes
        for i, col in enumerate(columns_to_plot):
            try:
                mode_amplitude = np.abs(
                    self.predhat_split[tsteps_range[0]:tsteps_range[-1], col]
                    + 1j * self.predhat_split[tsteps_range[0]:tsteps_range[-1], col + 1]
                )
            except Exception as exc:
                print(f"Failed to calculate mode amplitudes with model at {self.weights_file}")
                raise exc

            mode_amplitude_hist[:, i] = mode_amplitude

        return mode_amplitude_hist

    def moving_average(self, window_size=100):
        """Lazy loading moving average of each mode over time."""
        if self._mode_moving_average is None:
            self._mode_moving_average = np.zeros_like(self.mode_amplitude_hist, dtype=float)
            for i, mode_series in enumerate(self.mode_amplitude_hist.T):
                self._mode_moving_average[:, i] = (
                    pd.Series(mode_series).rolling(window=window_size, min_periods=1).mean().to_numpy()
                )

        return self._mode_moving_average

    def moving_std(self, window_size=100):
        """Lazy loading moving standard deviation of each mode over time."""
        if self._mode_moving_std is None:
            self._mode_moving_std = np.zeros_like(self.mode_amplitude_hist, dtype=float)
            for i, mode_series in enumerate(self.mode_amplitude_hist.T):
                self._mode_moving_std[:, i] = (
                    pd.Series(mode_series).rolling(window=window_size, min_periods=1).std(ddof=0).to_numpy()
                )

        return self._mode_moving_std

    def blow_up_tsteps(self, window_rad=50, r_min=0.99):
        """Lazy loading time steps of mode blow up."""
        if self._blow_up_tsteps is None:
            num_modes = self.mode_amplitude_hist.shape[1]
            total_steps = self.mode_amplitude_hist.shape[0]
            self._blow_up_tsteps = np.full(num_modes, total_steps, dtype=int)

            for i in range(num_modes):
                mode_series = self.mode_amplitude_hist[:, i]
                blown_up = False
                tstep = window_rad
                while tstep < total_steps - window_rad and not blown_up:
                    window = np.log(np.maximum(mode_series[tstep - window_rad : tstep + window_rad], 1e-12))
                    _, _, r, _, _ = stats.linregress(np.arange(len(window)), window)
                    if r > r_min:
                        self._blow_up_tsteps[i] = tstep + window_rad
                        blown_up = True
                    tstep += 20

        return self._blow_up_tsteps

    @property
    def initialization_description(self):
        return self.initialization_type.replace("_", " ").title()

    def save_mode_info(self, overwrite=False, save_predhat=True, save_mode_amplitudes=True):
        """Saves calculated information about Fourier mode amplitudes."""
        to_hdf_mode = "w" if overwrite else "a"

        if save_predhat:
            save_name = self.save_dirs["predhat_split_hists"]
            if self.predhat_split is not None:
                df = pd.DataFrame(self._predhat_split)
                df.to_hdf(save_name, key="df", mode=to_hdf_mode, complib="blosc", complevel=9)
            else:
                logger.warning(f"There is nothing to save at {save_name}")

        if save_mode_amplitudes:
            save_name = self.save_dirs["mode_amplitude_hists"]
            if self._mode_amplitude_hist is not None:
                df = pd.DataFrame(self._mode_amplitude_hist)
                df.to_hdf(save_name, key="df", mode=to_hdf_mode, complib="blosc", complevel=9)
            else:
                logger.warning(f"There is nothing to save at {save_name}")

    @cached_property
    def _calc_real(self):
        """Use ksfm2real to calculate u, ux, and uxx of the prediction uhat."""
        _, us, u_xs, u_xxs = ksfm2real(self.predhat_split.T, 22)
        us = us.T
        u_xs = u_xs.T
        u_xxs = u_xxs.T

        return {"us": us, "u_xs": u_xs, "u_xxs": u_xxs}

    @property
    def us(self):
        """Lazy loading us."""
        if self._us is None:
            self._us = self._calc_real["us"]
        return self._us

    @property
    def u_xs(self):
        """Lazy loading u."""
        if self._u_xs is None:
            self._u_xs = self._calc_real["u_xs"]
        return self._u_xs

    @property
    def u_xxs(self):
        """Lazy loading u."""
        if self._u_xxs is None:
            self._u_xxs = self._calc_real["u_xxs"]
        return self._u_xxs

    def _make_save_dirs(self, overwrite):
        dirs = {
            "mode_evolution_figs": "",
            "trajectory_figs": "",
            "pdf_figs": "",
            "predhat_split_hists": "",
            "mode_amplitude_hists": "",
        }
        for key in dirs:
            extension = ".png" if "figs" in key else (".h5" if "hists" in key else "")
            dirs[key] = Path(key) / f"{self.clean_weights_file.as_posix()}{extension}"
        for value in dirs.values():
            if not overwrite and value.exists():
                raise FileExistsError(
                    f"{value} already exists. Change OVERWRITE to True if you want to overwrite the existing version."
                )
            value.parent.mkdir(parents=True, exist_ok=True)
        return dirs

In [ ]:
# CREATE MODEL WRAPPERS AND LOAD MODELS

wrapped_models = []
for file in WEIGHTS_FILES:
    wrapped_models.append(ModelWrapper(file, SIMPLY_LOAD, OVERWRITE))

# sort the models by epoch number
wrapped_models.sort(key=lambda x: x.epoch)

# assign variables to be used for slideshow plotting
epoch_to_model_map = {wm.epoch: (idx, wm) for idx, wm in enumerate(wrapped_models)}
EPOCHS_TO_SHOW = [wrapped_model.epoch for wrapped_model in wrapped_models]

In [ ]:
# RUN PREDICTION METHOD & FOURIER MODE CALCULATIONS METHOD THEN SAVE TO H5

for wrapped_model in wrapped_models:
    # call on the predhat_split property for the first time to lazy load the prediction
    try:
        if not wrapped_model.simply_load:
            wrapped_model.predhat_split
    except Exception as e:
        print(f"Failed to make predictions with model at {wrapped_model.weights_file}")
        raise e

    #call on the mode_amplitude_hist property for the first time to lazy load the mode amplitude history and save the hd5s
    try:
        wrapped_model.mode_amplitude_hist

    except Exception as e:
        print(f"Failed to calculate mode amplitudes with model at {wrapped_model.weights_file}")
        raise e

    # calculate other statistics about mode amplitudes
    try:
        wrapped_model.moving_average()
        wrapped_model.moving_std()
        wrapped_model.blow_up_tsteps()
    except Exception as e:
        print(f"Failed to calculate mode amplitudes with model at {wrapped_model.weights_file}")
        raise e

    if not SIMPLY_LOAD: wrapped_model.save_mode_info()

In [ ]:
# SAVE & PLOT FOURIER MODES OVER TIME

# aesthetics setup
colormap = plt.cm.inferno
colors = [colormap(i) for i in np.linspace(0, 1, 32)]

# what and how much to plot
columns_to_plot = np.arange(0, 62, 2, dtype=int)

if SAVE_PICS:
    # Dictionary to store the absolute file path for each epoch
    epoch_to_image_path = {}

for idx, wrapped_model in enumerate(wrapped_models):
    tsteps_range = np.array([0, series_length_map[wrapped_model.equation]], dtype=int)
    epoch = wrapped_model.epoch
    if SAVE_PICS:
        # variables to keep track of where to save the plot
        file_path = wrapped_model.save_dirs['mode_evolution_figs']
        epoch_to_image_path[epoch] = str(file_path)
    
    # plot setup
    plt.figure(figsize=(12, 6))
    
    # plot the mode amplitudes
    for i, col in enumerate(columns_to_plot):
        plt.plot(
            np.arange(tsteps_range[0], tsteps_range[-1], 1), wrapped_model.mode_amplitude_hist[:, i],  
            color = colors[i], alpha=0.85, linewidth=1.5
        )

    # plot fanciness
    sm = plt.cm.ScalarMappable(cmap=colormap, norm=plt.Normalize(vmin=0, vmax=32))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=plt.gca())
    cbar.set_label('Mode Number (n)', fontsize=12, labelpad=10)

    # plot labels
    title = f'Model {idx+1}: {wrapped_model.clean_weights_file.as_posix().replace("-", " | ").replace("/", " | ")}'
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.title('Time Evolution of KSE Wave Modes', fontsize=10)
    plt.xlabel('Time ($t$) / quarter seconds', fontsize=12)
    plt.ylabel('Mode Amplitude  ($|\\hat{u}_i|$) / arb', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.4)
    _, default_ymax = plt.ylim()
    #plt.ylim(0,min(default_ymax, 5))
    plt.tight_layout()

    if SAVE_PICS:
        # Save it explicitly to disk
        plt.savefig(file_path, dpi=150) # Adjust DPI if you want smaller/larger files
        plt.close() # Close it immediately to free memory
    else:
        plt.show()

if SAVE_PICS:
    print("All plots saved successfully!")
    slideshow(epoch_to_image_path, EPOCHS_TO_SHOW)

In [ ]:
# PLOT TRAJECTORY PREDICTIONS

if SAVE_PICS:
    # Dictionary to store the absolute file path for each epoch
    epoch_to_image_path = {}

for wrapped_model in wrapped_models:
    tsteps_range = np.array([0, min(series_length_map[wrapped_model.equation], 2000)], dtype=int)
    epoch = wrapped_model.epoch
    if SAVE_PICS:
        file_path = wrapped_model.save_dirs['trajectory_figs']
        epoch_to_image_path[epoch] = str(file_path)

    plt.figure(figsize=(12,6))
    plt.imshow(np.transpose(wrapped_model.us[tsteps_range[0]:tsteps_range[-1],:]), cmap='bwr', interpolation='nearest', aspect='auto', extent=[tsteps_range[0],tsteps_range[-1],-11,11])

    title = f'Model {idx+1}: {wrapped_model.clean_weights_file.as_posix().replace("-", " | ").replace("/", " | ")}'
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.title('Trajectory Prediction of KSE', fontsize=14, fontweight='bold')
    plt.xlabel('Time ($t$) / quarter seconds', fontsize=12)
    plt.ylabel('Position ($x$) / meters')
    plt.colorbar(label=r'$\tilde{u}$')

    if SAVE_PICS:
        plt.savefig(file_path, dpi=150)
        plt.close()
    else:
        plt.show()

if SAVE_PICS:
    print("All plots saved successfully!")
    slideshow(epoch_to_image_path, EPOCHS_TO_SHOW)

In [ ]:
# calculate and plot the joint probability mass function of u_x and u_xx

if SAVE_PICS:
    # Dictionary to store the absolute file path for each epoch
    epoch_to_image_path = {}

for wrapped_model in wrapped_models:
    if series_length_map[wrapped_model.equation] == 40000:
        tsteps_range = np.array([5000, -1], dtype=int)        
    else:
        tsteps_range = np.array([0, min(series_length_map[wrapped_model.equation], 2000)], dtype=int)
    epoch = wrapped_model.epoch
    if SAVE_PICS:
        file_path = wrapped_model.save_dirs['pdf_figs']
        epoch_to_image_path[epoch] = str(file_path)

    u_xs_flat = wrapped_model.u_xs[tsteps_range[0]:tsteps_range[-1]].flatten()
    u_xxs_flat = wrapped_model.u_xxs[tsteps_range[0]:tsteps_range[-1]].flatten()

    plt.figure(figsize=(8, 6))

    # Plot the 2D histogram
    # bins=100 splits the space into a 100x100 grid; density=True normalizes it to a PDF
    plt.hist2d(u_xs_flat, u_xxs_flat, bins=500, cmap='inferno', density=True, norm='log')

    plt.colorbar(label='Probability Density')

    title = f'Model {idx+1}: {wrapped_model.clean_weights_file.as_posix().replace("-", " | ").replace("/", " | ")}'
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.title('Joint PDF of $u_x$ and $u_{xx}$')
    plt.xlabel('$u_x$ (First Spatial Derivative)')
    plt.ylabel('$u_{xx}$ (Second Spatial Derivative)')

    plt.grid(True, alpha=0.3)
    #plt.axis([-2, 4, -4, 4])

    if SAVE_PICS:
        plt.savefig(file_path, dpi=150)
        plt.close()
    else:
        plt.show()

if SAVE_PICS:
    print("All plots saved successfully!")
    slideshow(epoch_to_image_path, EPOCHS_TO_SHOW)

In [ ]:
# FIND RUNAWAY TIMES (NOT INCORPORATED IN OOP BC RAN OUT OF TIME)

# Load Ground Truth once
ground_truth_mode_evolution = pd.read_hdf(
    '../mode_evolution_csvs/mode_amplitude_histories/ground_state.h5', key='df'
).to_numpy()
mode_thresholds = [0.17751612417578, 0.1256623675230528, 0.39757182529601565, 0.0895254424892934, 0.11159391719837072, 0.0215877523953181, 0.02363306740905142, 0.0038979222327725402, 
                   0.003962448745272087, 0.0007095601887166294, 0.0006048335869692214, 0.00011436854105493195, 8.642565789576936e-05, 1.7734005053600777e-05, 1.1730643780133875e-05, 2.6952149570312987e-06, 
                   1.5380486372624384e-06, 3.907272767673509e-07, 2.0073318625629434e-07, 5.431481212921456e-08, 2.5850534214683195e-08, 7.4309949027561705e-09, 3.2937977653240122e-09, 9.919045837917934e-10, 
                   4.1749942360788535e-10, 1.301410136023945e-10, 5.2536238054247803e-11, 1.6766480350422337e-11, 6.597533037727205e-12, 2.1527683249423827e-12, 8.262748093781074e-13]

runaway_times = np.zeros((len(wrapped_models), 31))

for i, wrapped_model in enumerate(wrapped_models):

    for j in range(ground_truth_mode_evolution.shape[1]):
        gt_mode = ground_truth_mode_evolution[:, j]
        pred_mode = wrapped_model.mode_amplitude_hist[:, j]

        # Compute runaway timestep for this specific mode
        # t_runaway = find_attractor_runaway_wasserstein(
        #     predictions=pred_mode,
        #     ground_truth=gt_mode,
        #     ignore_first=5,
        #     window_rad=200,
        #     distance_threshold=mode_thresholds[j]
        # )
        
        t_runaway = find_attractor_runaway_ztest(
            predictions=pred_mode,
            ground_truth=gt_mode,
            ignore_first=5,
            window_rad=200,
            z_threshold=2.81,  # corresponds to a p-val of 0.005
        )

        runaway_times[i][j] = t_runaway

fig, ax = plt.subplots(figsize=(10,6))

colormap = plt.cm.cividis
colors = [colormap(i) for i in np.linspace(0, 0.5, len(runaway_times))]

for idx, row in enumerate(runaway_times):
    normalized_print = "normalized" if wrapped_models[idx].normalized else "unnormalized"
    ax.scatter(np.arange(1,32,1), row[:], color=colors[idx], alpha=0.75, linewidths=0, label=f"{normalized_print}")

# sm = plt.cm.ScalarMappable(cmap=colormap, norm=plt.Normalize(vmin=0, vmax=len(runaway_times)))
# sm.set_array([])
# cbar = plt.colorbar(sm, ax=plt.gca())
# cbar.set_label("Model index", fontsize=12, labelpad=10)

ax.set_yscale('log')
#ax.set_title("Mode runaway for models trained in Fourier space with various initial standard deviations", fontsize=18)
ax.set_ylabel("Timesteps until runaway", fontsize=14)
ax.set_xlabel("Mode number (n)", fontsize=14)
ax.legend(fontsize=14)
plt.tight_layout()

plt.show()